# E-commerce Fulfilment & Delivery Performance Analysis

## Notebook 03: Exploratory Data Analysis

This notebook focuses on exploring the analysis-ready Olist e-commerce dataset.

The purpose of this notebook is to:

- Review the cleaned master dataset
- Calculate key fulfilment and delivery KPIs
- Analyse late delivery performance
- Explore review score patterns
- Compare freight cost by product category
- Identify useful insights for SQL, Power BI and business recommendations

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Define file paths

processed_data_path = Path("../data/processed")
analysis_file = processed_data_path / "analysis_ready_orders.csv"

print("Analysis file exists:", analysis_file.exists())

In [ ]:
# Load analysis-ready dataset

orders = pd.read_csv(analysis_file)

print("Dataset loaded successfully.")
print("Rows:", orders.shape[0])
print("Columns:", orders.shape[1])

In [ ]:
# Preview the first rows

orders.head()

In [ ]:
# Check column names

orders.columns.tolist()

In [ ]:
# Check data types

orders.info()

## Key Performance Indicators

In this section, I will calculate the main KPIs for delivery and fulfilment performance.

These KPIs help summarise the overall performance of the delivered orders in the dataset.

In [ ]:
# Calculate key KPIs

total_orders = len(orders)
late_orders = orders["is_late_delivery"].sum()
late_delivery_rate = late_orders / total_orders

average_delivery_days = orders["delivery_days"].mean()
average_delay_days = orders["delay_days"].mean()
average_review_score = orders["avg_review_score"].mean()
average_freight_value = orders["total_freight_value"].mean()
average_order_value = orders["total_item_value"].mean()

kpi_summary = pd.DataFrame([
    {"KPI": "Total Delivered Orders", "Value": total_orders},
    {"KPI": "Late Delivered Orders", "Value": late_orders},
    {"KPI": "Late Delivery Rate", "Value": f"{late_delivery_rate:.2%}"},
    {"KPI": "Average Delivery Days", "Value": round(average_delivery_days, 2)},
    {"KPI": "Average Delay Days", "Value": round(average_delay_days, 2)},
    {"KPI": "Average Review Score", "Value": round(average_review_score, 2)},
    {"KPI": "Average Freight Value", "Value": round(average_freight_value, 2)},
    {"KPI": "Average Item Value", "Value": round(average_order_value, 2)}
])

kpi_summary

## Late Delivery and Review Score

This section checks whether late deliveries appear to be linked with lower customer review scores.

This is important because delivery performance can directly affect customer satisfaction.

In [ ]:
# Compare review score for late and on-time deliveries

review_by_delivery_status = (
    orders
    .groupby("is_late_delivery")
    .agg(
        order_count=("order_id", "count"),
        average_review_score=("avg_review_score", "mean"),
        average_delivery_days=("delivery_days", "mean"),
        average_delay_days=("delay_days", "mean")
    )
    .reset_index()
)

review_by_delivery_status["delivery_status"] = review_by_delivery_status["is_late_delivery"].map({
    False: "On time or early",
    True: "Late"
})

review_by_delivery_status

## Customer State Analysis

This section shows where delivered orders are concentrated by customer state.

This can help identify regions with higher order volumes and possible delivery pressure.

In [ ]:
# Top customer states by order volume

state_summary = (
    orders
    .groupby("customer_state")
    .agg(
        order_count=("order_id", "count"),
        late_orders=("is_late_delivery", "sum"),
        average_review_score=("avg_review_score", "mean"),
        average_delivery_days=("delivery_days", "mean"),
        average_freight_value=("total_freight_value", "mean")
    )
    .reset_index()
)

state_summary["late_delivery_rate"] = state_summary["late_orders"] / state_summary["order_count"]

state_summary.sort_values(by="order_count", ascending=False).head(10)

## Product Category Analysis

This section explores delivery, freight and review patterns by product category.

This can help identify whether some product categories are linked with higher freight costs or poorer delivery outcomes.

In [ ]:
# Product category summary

category_summary = (
    orders
    .groupby("main_product_category")
    .agg(
        order_count=("order_id", "count"),
        late_orders=("is_late_delivery", "sum"),
        average_review_score=("avg_review_score", "mean"),
        average_delivery_days=("delivery_days", "mean"),
        average_freight_value=("total_freight_value", "mean"),
        average_item_value=("total_item_value", "mean")
    )
    .reset_index()
)

category_summary["late_delivery_rate"] = category_summary["late_orders"] / category_summary["order_count"]

category_summary.sort_values(by="order_count", ascending=False).head(10)

## Initial Visual Exploration

The following charts provide a quick visual overview of key fulfilment and delivery patterns.

More polished dashboard visuals will be created later in Power BI.

In [ ]:
# Bar chart: late vs on-time orders

delivery_status_counts = orders["is_late_delivery"].value_counts().rename(index={
    False: "On time or early",
    True: "Late"
})

delivery_status_counts.plot(kind="bar")
plt.title("Delivered Orders: Late vs On-Time")
plt.xlabel("Delivery Status")
plt.ylabel("Number of Orders")
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Bar chart: average review score by delivery status

review_chart_data = review_by_delivery_status.set_index("delivery_status")["average_review_score"]

review_chart_data.plot(kind="bar")
plt.title("Average Review Score by Delivery Status")
plt.xlabel("Delivery Status")
plt.ylabel("Average Review Score")
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Top 10 product categories by order count

top_categories = category_summary.sort_values(by="order_count", ascending=False).head(10)

top_categories.set_index("main_product_category")["order_count"].plot(kind="bar")
plt.title("Top 10 Product Categories by Order Count")
plt.xlabel("Product Category")
plt.ylabel("Number of Orders")
plt.xticks(rotation=45, ha="right")
plt.show()

## Export EDA Summary Tables

The summary tables created in this notebook will be exported to the processed data folder.

These can be reused later in reports, SQL comparison work or Power BI.

In [ ]:
# Export EDA summary tables

kpi_summary.to_csv(processed_data_path / "kpi_summary.csv", index=False)
review_by_delivery_status.to_csv(processed_data_path / "review_by_delivery_status.csv", index=False)
state_summary.to_csv(processed_data_path / "state_summary.csv", index=False)
category_summary.to_csv(processed_data_path / "category_summary.csv", index=False)

print("EDA summary tables exported successfully.")

In [ ]:
# Check exported files

for file_name in [
    "kpi_summary.csv",
    "review_by_delivery_status.csv",
    "state_summary.csv",
    "category_summary.csv"
]:
    file_path = processed_data_path / file_name
    print(file_name, file_path.exists())